# SautiCivic Bridge — OpenAI Whisper large-v3 Benchmark (Tier B Public Corpus)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spyder0000/Sauticivic/blob/main/bench/whisper_tier_b_colab.ipynb)

This notebook runs **OpenAI Whisper large-v3** on Google Colab GPU to transcribe the **60 Tier B public benchmark clips** across 4 language pairs:
- **AfriSwitch** (30 clips: Pidgin-English, Yoruba-English, Hausa-English)
- **FLEURS** (15 clips: Hausa, Yoruba)
- **AfriSpeech-200** (15 clips: Nigerian-accented English)

### Recommended Colab Setup:
1. Go to **Runtime** > **Change runtime type** in the top menu.
2. Select **T4 GPU** (or A100/V100) under *Hardware Accelerator*.
3. Run the cells in order.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Clone Repository & Setup Working Directory

In [ ]:
import os
from pathlib import Path

if not os.path.exists("Sauticivic"):
    print("Cloning Sauticivic repository...")
    !git clone https://github.com/Spyder0000/Sauticivic.git

%cd Sauticivic
!git pull origin main
print(f"\nWorking directory: {os.getcwd()}")

## 3. Install Dependencies
Installs `openai-whisper`, `soundfile`, `ffmpeg`, and HuggingFace dependencies.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q openai-whisper soundfile datasets huggingface_hub python-dotenv

## 4. Acquire / Load Tier B Audio Files

Choose **Option A** (Direct download from HuggingFace on Colab) or **Option B** (Mount Google Drive).

In [ ]:
# === OPTION A: Live Ingestion directly inside Colab (RECOMMENDED) ===
# Downloads and standardizes all 60 clips directly from HuggingFace in ~2 minutes
import os

# If you have a HuggingFace token, set it here (optional for public datasets):
# os.environ["HF_TOKEN"] = "hf_..."

!python bench/corpus/tier_b_public/ingest_tier_b.py --dataset all

In [ ]:
# === OPTION B: Copy from Google Drive (Alternative) ===
# Uncomment if you have the audio files in your Google Drive
# from google.colab import drive
# drive.mount("/content/drive", force_remount=False)
# !mkdir -p bench/corpus/tier_b_public
# !cp -r /content/drive/MyDrive/tier_b_public/* bench/corpus/tier_b_public/

In [ ]:
# === Verify Audio Clips Discovery ===
from pathlib import Path
corpus_dir = Path("bench/corpus/tier_b_public")
audio_files = sorted(list(corpus_dir.rglob("*.wav")))

print(f"Total Tier B audio clips found: {len(audio_files)}/60")
if len(audio_files) > 0:
    print("First 5 clips detected:")
    for f in audio_files[:5]:
        print(f"  - {f.relative_to(corpus_dir)}")
else:
    print("⚠ No audio files found. Please run Option A or Option B above.")

## 5. Run Whisper large-v3 Transcription

Transcribes all 60 audio clips using Whisper `large-v3` on GPU and writes per-clip JSON records to `bench/results/transcripts/tier_b/whisper/`.

In [ ]:
!python bench/models/run_whisper.py \
    --corpus bench/corpus/tier_b_public \
    --output-dir bench/results/transcripts/tier_b \
    --model large-v3

## 6. Inspect Results & Export Transcripts

Download the generated JSON transcripts as a zip file to place into your local repository.

In [ ]:
import json
from pathlib import Path
from google.colab import files

out_dir = Path("bench/results/transcripts/tier_b/whisper")
transcripts = list(out_dir.glob("*.json"))
print(f"Total Whisper transcript files generated: {len(transcripts)}/60\n")

# Preview first 3 transcripts
for p in sorted(transcripts)[:3]:
    data = json.loads(p.read_text(encoding="utf-8"))
    print(f"[{data['clip_id']}] ({data.get('language')}): {data['transcript'][:80]}...")

# Package as a zip file for instant download
!zip -r whisper_tier_b_transcripts.zip bench/results/transcripts/tier_b/whisper/

print("\nDownloading zip file...")
files.download("whisper_tier_b_transcripts.zip")
print("✓ Done! Extract the downloaded zip into your local repository at:")
print("  bench/results/transcripts/tier_b/whisper/")